# MNIST Echo-State Scratchpad

This notebook uses one MNIST pixel out of each 2x2 block as a 14x14 input layer, then drives a 28x28 reservoir layer. The reservoir is a true 2x spatial scale-up of the input grid, so it has 4x as many neurons as the input layer. Input-to-reservoir wiring is locality-biased in the upsampled coordinate frame, like a random reverse convolution.

The live section includes online supervised training of the readout. Digit buttons can present and train one labeled example, and the batch/auto-train controls run online RLS updates from random MNIST examples. RLS is closer to the usual echo-state-network readout than softmax SGD, because each mini-batch updates a linear least-squares decoder while the reservoir supplies the dynamics.
Each reservoir neuron samples its own `Delta`, `Eta`, `J`, and current-decay parameters, so the layer is heterogeneous rather than made from identical cells.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from urllib.request import urlretrieve
import gzip
import struct
import threading
import time

import cupy as cp
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from tqdm.auto import tqdm

cp.cuda.Device(0).use()
print(f"CuPy {cp.__version__}; CUDA devices: {cp.cuda.runtime.getDeviceCount()}")

## MNIST Loader

No sklearn/torch/tensorflow dependency is needed. The IDX files are downloaded once into `.spikecore.cache/mnist` and parsed directly.

In [ ]:
MNIST_CACHE = Path(".spikecore.cache/mnist")
MNIST_BASE_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"
MNIST_FILES = {
    "train_images": "train-images-idx3-ubyte.gz",
    "train_labels": "train-labels-idx1-ubyte.gz",
    "test_images": "t10k-images-idx3-ubyte.gz",
    "test_labels": "t10k-labels-idx1-ubyte.gz",
}


def ensure_mnist(cache_dir: Path = MNIST_CACHE):
    cache_dir.mkdir(parents=True, exist_ok=True)
    for filename in MNIST_FILES.values():
        path = cache_dir / filename
        if not path.exists():
            print(f"Downloading {filename}...")
            urlretrieve(MNIST_BASE_URL + filename, path)


def read_mnist_images(path: Path):
    with gzip.open(path, "rb") as f:
        magic, count, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError(f"Unexpected image file magic: {magic}")
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(count, rows, cols)


def read_mnist_labels(path: Path):
    with gzip.open(path, "rb") as f:
        magic, count = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError(f"Unexpected label file magic: {magic}")
        return np.frombuffer(f.read(), dtype=np.uint8)


def load_mnist(cache_dir: Path = MNIST_CACHE):
    ensure_mnist(cache_dir)
    train_images = read_mnist_images(cache_dir / MNIST_FILES["train_images"])
    train_labels = read_mnist_labels(cache_dir / MNIST_FILES["train_labels"])
    test_images = read_mnist_images(cache_dir / MNIST_FILES["test_images"])
    test_labels = read_mnist_labels(cache_dir / MNIST_FILES["test_labels"])
    return train_images, train_labels, test_images, test_labels


def select_balanced(images, labels, per_digit: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    selected = []
    for digit in range(10):
        choices = np.flatnonzero(labels == digit)
        selected.extend(rng.choice(choices, size=per_digit, replace=False))
    selected = np.asarray(selected)
    rng.shuffle(selected)
    return images[selected], labels[selected]


def sample_balanced(images, labels, per_digit: int, rng):
    selected = []
    for digit in range(10):
        choices = np.flatnonzero(labels == digit)
        selected.extend(rng.choice(choices, size=per_digit, replace=False))
    selected = np.asarray(selected)
    rng.shuffle(selected)
    return images[selected], labels[selected]


def sample_random_labeled(images, labels, count: int, rng):
    # Balanced-ish random labels keep online batches from being dominated by common digits.
    digits = np.arange(10, dtype=np.uint8)
    requested = np.resize(digits, count)
    rng.shuffle(requested)
    indices = []
    for digit in requested:
        choices = np.flatnonzero(labels == digit)
        indices.append(int(rng.choice(choices)))
    indices = np.asarray(indices)
    return images[indices], labels[indices]


def input_from_images(images):
    # One input neuron for one pixel in each 2x2 block: 28x28 -> 14x14.
    return (images[:, ::2, ::2].astype(np.float32) / 255.0).reshape(len(images), -1)


def one_hot(labels, classes: int = 10):
    return np.eye(classes, dtype=np.float32)[np.asarray(labels, dtype=np.int64)]


train_images_all, train_labels_all, test_images_all, test_labels_all = load_mnist()
print(train_images_all.shape, test_images_all.shape)

## Configuration

The reservoir shape is the input shape scaled by 2 in both axes: 14x14 becomes 28x28. Local input wiring maps each reservoir neuron back into low-resolution input coordinates and samples nearby pixels, which preserves spatial structure while allowing random local mixing.
Dynamic parameter stds below control per-neuron heterogeneity. Set a std to `0.0` to recover the old constant value for that parameter.


In [ ]:
@dataclass
class ReservoirConfig:
    input_shape: tuple[int, int] = (14, 14)
    reservoir_shape: tuple[int, int] = (28, 28)
    seed: int = 3

    # Local random connectivity. Larger sigma makes connections less local.
    rec_fan_in: int = 48
    input_fan_in: int = 9
    rec_sigma: float = 3.0
    input_sigma: float = 0.85
    spectral_radius: float = 0.85
    input_weight_scale: float = 1.2

    # Main plasticity path: recurrent rate-STDP/Oja for reservoir -> reservoir synapses.
    # It preserves the initial local sparse mask and renormalizes each post-neuron row.
    rec_plastic_lr: float = 2e-4
    rec_plastic_ltp: float = 1.0
    rec_plastic_ltd: float = 0.75
    rec_plastic_oja: float = 1.0
    rec_plastic_decay: float = 1e-4
    rec_plastic_clip: float = 1.0
    rec_plastic_normalize: bool = True

    # Optional secondary rule for input -> reservoir synapses.
    # The update preserves the initial local sparse mask and renormalizes each row.
    input_plastic_lr: float = 2e-3
    input_plastic_oja: float = 1.0
    input_plastic_decay: float = 1e-4
    input_plastic_clip: float = 2.0
    input_plastic_normalize: bool = True

    # MPR-like rate/voltage dynamics for the reservoir layer.
    # Delta/Eta/J/current_decay are now per-neuron sampled values centered on these means.
    dt: float = 0.01
    Delta: float = 1.0
    Eta: float = -2.5
    J: float = 0.0
    current_decay: float = 0.2
    randomize_cell_dynamics: bool = True
    Delta_std: float = 0.15
    Eta_std: float = 0.35
    J_std: float = 0.0
    current_decay_std: float = 0.03
    Delta_clip: tuple[float, float] = (0.05, 4.0)
    Eta_clip: tuple[float, float] = (-8.0, 2.0)
    J_clip: tuple[float, float] = (-3.0, 3.0)
    current_decay_clip: tuple[float, float] = (0.0, 0.95)

    # Image presentation timing.
    pre_steps: int = 2
    on_steps: int = 28
    off_steps: int = 10
    readout_start: int = 12
    amplitude: float = 4.0

    # Numeric guards.
    r_clip: tuple[float, float] = (0.0, 20.0)
    v_clip: tuple[float, float] = (-50.0, 50.0)
    i_clip: tuple[float, float] = (-50.0, 50.0)


@dataclass
class ProbeConfig:
    train_per_digit: int = 500
    test_per_digit: int = 100
    ridge: float = 1e-2
    batch_size: int = 128
    # Softmax-SGD baseline. Useful as a comparison against the RLS readout.
    online_lr: float = 5e-2
    online_l2: float = 1e-4
    online_epochs: int = 5
    online_batch_size: int = 64

    # Online ridge/RLS readout used by the live view.
    # rls_delta is the initial ridge prior: smaller values adapt faster and regularize less.
    rls_delta: float = 1e-2
    rls_forgetting: float = 1.0
    rls_epochs: int = 1
    rls_batch_size: int = 64


RESERVOIR_CONFIG = ReservoirConfig()
PROBE_CONFIG = ProbeConfig()

n_input = int(np.prod(RESERVOIR_CONFIG.input_shape))
n_reservoir = int(np.prod(RESERVOIR_CONFIG.reservoir_shape))
print(f"input neurons: {n_input}; reservoir neurons: {n_reservoir}; ratio: {n_reservoir / n_input:.1f}x")


## Reservoir Dynamics

The readout uses `R`, the reservoir rate variable. `V` remains an internal state variable for the dynamics only.

In [ ]:
rk4_step_kernel = cp.ElementwiseKernel(
    "float32 R, float32 V, float32 I, float32 dt, float32 Delta, float32 Eta, float32 J",
    "float32 R_out, float32 V_out",
    r"""
    const float pi = 3.14159265358979323846f;
    const float pi2 = pi * pi;

    float k1R = Delta / pi + 2.0f * R * V;
    float k1V = V * V + Eta + J * R + I - pi2 * R * R;

    float R2 = R + 0.5f * dt * k1R;
    float V2 = V + 0.5f * dt * k1V;
    float k2R = Delta / pi + 2.0f * R2 * V2;
    float k2V = V2 * V2 + Eta + J * R2 + I - pi2 * R2 * R2;

    float R3 = R + 0.5f * dt * k2R;
    float V3 = V + 0.5f * dt * k2V;
    float k3R = Delta / pi + 2.0f * R3 * V3;
    float k3V = V3 * V3 + Eta + J * R3 + I - pi2 * R3 * R3;

    float R4 = R + dt * k3R;
    float V4 = V + dt * k3V;
    float k4R = Delta / pi + 2.0f * R4 * V4;
    float k4V = V4 * V4 + Eta + J * R4 + I - pi2 * R4 * R4;

    float fac = dt / 6.0f;
    R_out = R + fac * (k1R + 2.0f * k2R + 2.0f * k3R + k4R);
    V_out = V + fac * (k1V + 2.0f * k2V + 2.0f * k3V + k4V);
    """,
    "mnist_echo_mpr_rk4_f32_v2",
)


class MNISTEchoReservoir:
    def __init__(self, config: ReservoirConfig):
        self.config = config
        self.n_input = int(np.prod(config.input_shape))
        self.n_reservoir = int(np.prod(config.reservoir_shape))
        expected = (2 * config.input_shape[0], 2 * config.input_shape[1])
        if config.reservoir_shape != expected:
            raise ValueError(f"reservoir_shape should be a 2x square scale-up: expected {expected}")
        self._build_weights()
        self._build_cell_dynamics()
        self.reset_state(batch_size=1)

    def _positions(self, shape, scale_to_input: float):
        h, w = shape
        yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
        return np.column_stack([yy.ravel() * scale_to_input, xx.ravel() * scale_to_input])

    def _local_random_matrix(self, src_pos, dst_pos, fan_in, sigma, scale, rng, allow_self=True):
        W = np.zeros((len(dst_pos), len(src_pos)), dtype=np.float32)
        fan_in = min(int(fan_in), len(src_pos))
        for row, pos in enumerate(dst_pos):
            dist2 = np.sum((src_pos - pos) ** 2, axis=1)
            prob = np.exp(-dist2 / (2.0 * sigma * sigma)) + 0.001
            if not allow_self and len(src_pos) == len(dst_pos):
                prob[row] = 0.0
            prob /= prob.sum()
            cols = rng.choice(len(src_pos), size=fan_in, replace=False, p=prob)
            W[row, cols] = rng.normal(0.0, scale / np.sqrt(fan_in), size=fan_in)
        return W

    def _build_weights(self):
        cfg = self.config
        rng = np.random.default_rng(cfg.seed)

        # Input pixels live on integer 14x14 coordinates. Reservoir neurons live on a 28x28 grid
        # but are mapped back to that coordinate frame by 0.5, like an upsampled/deconvolution grid.
        self.input_positions = self._positions(cfg.input_shape, scale_to_input=1.0)
        self.reservoir_positions = self._positions(cfg.reservoir_shape, scale_to_input=0.5)

        W_in = self._local_random_matrix(
            self.input_positions,
            self.reservoir_positions,
            cfg.input_fan_in,
            cfg.input_sigma,
            scale=cfg.input_weight_scale,
            rng=rng,
            allow_self=True,
        )
        W_rec = self._local_random_matrix(
            self.reservoir_positions,
            self.reservoir_positions,
            cfg.rec_fan_in,
            cfg.rec_sigma,
            scale=1.0,
            rng=rng,
            allow_self=False,
        )

        radius = max(float(np.max(np.abs(np.linalg.eigvals(W_rec.astype(np.float64))))), 1e-6)
        W_rec = (W_rec / radius * cfg.spectral_radius).astype(np.float32)

        self.W_in = cp.asarray(W_in)
        self.W_rec = cp.asarray(W_rec)
        self.W_in_mask = cp.asarray(W_in != 0.0)
        self.W_rec_mask = cp.asarray(W_rec != 0.0)
        self.W_in_target_norm = cp.sqrt(cp.sum(self.W_in * self.W_in, axis=1, keepdims=True)) + cp.float32(1e-6)
        self.W_rec_target_norm = cp.sqrt(cp.sum(self.W_rec * self.W_rec, axis=1, keepdims=True)) + cp.float32(1e-6)

    def _sample_cell_parameter(self, rng, mean, std, clip):
        if (not self.config.randomize_cell_dynamics) or std == 0:
            values = np.full(self.n_reservoir, mean, dtype=np.float32)
        else:
            values = rng.normal(mean, std, size=self.n_reservoir).astype(np.float32)
        return np.clip(values, clip[0], clip[1]).astype(np.float32)

    def _build_cell_dynamics(self):
        cfg = self.config
        rng = np.random.default_rng(cfg.seed + 10_003)
        self.Delta = cp.asarray(
            self._sample_cell_parameter(rng, cfg.Delta, cfg.Delta_std, cfg.Delta_clip)
        )[:, None]
        self.Eta = cp.asarray(
            self._sample_cell_parameter(rng, cfg.Eta, cfg.Eta_std, cfg.Eta_clip)
        )[:, None]
        self.J = cp.asarray(
            self._sample_cell_parameter(rng, cfg.J, cfg.J_std, cfg.J_clip)
        )[:, None]
        self.current_decay = cp.asarray(
            self._sample_cell_parameter(rng, cfg.current_decay, cfg.current_decay_std, cfg.current_decay_clip)
        )[:, None]

    def resample_cell_dynamics(self, seed: int | None = None):
        if seed is not None:
            old_seed = self.config.seed
            self.config.seed = int(seed)
            self._build_cell_dynamics()
            self.config.seed = old_seed
        else:
            self._build_cell_dynamics()
        return self.cell_dynamic_stats()

    def cell_dynamic_stats(self):
        def stats(x):
            x = x.ravel()
            return {
                "mean": float(cp.mean(x).get()),
                "std": float(cp.std(x).get()),
                "min": float(cp.min(x).get()),
                "max": float(cp.max(x).get()),
            }
        return {
            "Delta": stats(self.Delta),
            "Eta": stats(self.Eta),
            "J": stats(self.J),
            "current_decay": stats(self.current_decay),
        }

    def reset_state(self, batch_size: int = 1):
        self.R = cp.zeros((self.n_reservoir, batch_size), dtype=cp.float32)
        self.V = cp.zeros_like(self.R)
        self.I = cp.zeros_like(self.R)
        self.t = 0.0
        self.tick = 0

    def envelope(self, step: int, on_steps: int | None = None):
        on_steps = self.config.on_steps if on_steps is None else int(on_steps)
        if on_steps <= 1:
            return 1.0
        phase = step / (on_steps - 1)
        return 0.5 - 0.5 * np.cos(2.0 * np.pi * phase)

    def step_state(self, R, V, I, U=None, amplitude: float | None = None):
        cfg = self.config
        amp = cfg.amplitude if amplitude is None else float(amplitude)
        I = self.current_decay * I + self.W_rec @ R
        if U is not None:
            I = I + self.W_in @ (cp.float32(amp) * U)
        cp.clip(I, cfg.i_clip[0], cfg.i_clip[1], out=I)

        R, V = rk4_step_kernel(
            R,
            V,
            I,
            cp.float32(cfg.dt),
            self.Delta,
            self.Eta,
            self.J,
        )
        cp.clip(R, cfg.r_clip[0], cfg.r_clip[1], out=R)
        cp.clip(V, cfg.v_clip[0], cfg.v_clip[1], out=V)
        return R, V, I

    def collect_features(
        self,
        input_vectors: np.ndarray,
        batch_size: int = 128,
        amplitude: float | None = None,
        pre_steps: int | None = None,
        on_steps: int | None = None,
        off_steps: int | None = None,
        readout_start: int | None = None,
        progress_desc: str | None = None,
    ):
        cfg = self.config
        pre_steps = cfg.pre_steps if pre_steps is None else int(pre_steps)
        on_steps = cfg.on_steps if on_steps is None else int(on_steps)
        off_steps = cfg.off_steps if off_steps is None else int(off_steps)
        readout_start = cfg.readout_start if readout_start is None else int(readout_start)
        total_steps = pre_steps + on_steps + off_steps
        amp = cfg.amplitude if amplitude is None else float(amplitude)

        X = cp.asarray(input_vectors.astype(np.float32))
        features = []
        batch_starts = range(0, X.shape[0], batch_size)
        if progress_desc is not None:
            batch_starts = tqdm(
                batch_starts,
                total=(X.shape[0] + batch_size - 1) // batch_size,
                desc=progress_desc,
                leave=False,
            )
        for start in batch_starts:
            U_image = X[start : start + batch_size].T
            batch = U_image.shape[1]
            R = cp.zeros((self.n_reservoir, batch), dtype=cp.float32)
            V = cp.zeros_like(R)
            I = cp.zeros_like(R)
            R_accum = cp.zeros_like(R)
            readout_count = 0

            for step in range(total_steps):
                if pre_steps <= step < pre_steps + on_steps:
                    pulse_step = step - pre_steps
                    env = self.envelope(pulse_step, on_steps)
                    U = cp.float32(env) * U_image
                else:
                    U = None

                R, V, I = self.step_state(R, V, I, U=U, amplitude=amp)

                if step >= readout_start:
                    R_accum += R
                    readout_count += 1

            features.append(cp.asnumpy((R_accum / max(1, readout_count)).T))

        return np.vstack(features)



    def recurrent_weight_stats(self):
        active = self.W_rec[self.W_rec_mask]
        row_norm = cp.sqrt(cp.sum(self.W_rec * self.W_rec, axis=1))
        return {
            "active_mean": float(cp.mean(active).get()),
            "active_std": float(cp.std(active).get()),
            "active_min": float(cp.min(active).get()),
            "active_max": float(cp.max(active).get()),
            "row_norm_mean": float(cp.mean(row_norm).get()),
            "row_norm_std": float(cp.std(row_norm).get()),
        }

    def apply_recurrent_plasticity(
        self,
        pre_rates,
        post_rates,
        lr: float | None = None,
        ltp: float | None = None,
        ltd: float | None = None,
        oja: float | None = None,
        decay: float | None = None,
        clip: float | None = None,
        normalize: bool | None = None,
        return_stats: bool = False,
    ):
        """Temporally asymmetric rate-STDP/Oja update for reservoir recurrent weights.

        W_rec[row, col] maps pre-synaptic reservoir neuron col to post-synaptic neuron row.
        `pre_rates` should be the earlier reservoir rates and `post_rates` the later rates.
        With averaged features, passing the same array for both is a conservative Hebbian/Oja update.
        """
        cfg = self.config
        lr = cfg.rec_plastic_lr if lr is None else float(lr)
        ltp = cfg.rec_plastic_ltp if ltp is None else float(ltp)
        ltd = cfg.rec_plastic_ltd if ltd is None else float(ltd)
        oja = cfg.rec_plastic_oja if oja is None else float(oja)
        decay = cfg.rec_plastic_decay if decay is None else float(decay)
        clip = cfg.rec_plastic_clip if clip is None else float(clip)
        normalize = cfg.rec_plastic_normalize if normalize is None else bool(normalize)
        if lr == 0:
            return self.recurrent_weight_stats() if return_stats else None

        pre = cp.asarray(pre_rates, dtype=cp.float32)
        post = cp.asarray(post_rates, dtype=cp.float32)
        if pre.ndim == 1:
            pre = pre[None, :]
        if post.ndim == 1:
            post = post[None, :]
        if pre.shape != post.shape or pre.shape[1] != self.n_reservoir:
            raise ValueError("pre_rates and post_rates must both have shape (batch, n_reservoir)")

        # Per-example centering prevents the baseline rate from dominating the update.
        pre_c = pre - cp.mean(pre, axis=1, keepdims=True)
        post_c = post - cp.mean(post, axis=1, keepdims=True)
        batch = cp.float32(pre.shape[0])

        # LTP: pre before post. LTD: post-before-pre approximation using the reverse temporal pair.
        ltp_term = (post_c.T @ pre_c) / batch
        ltd_term = (pre_c.T @ post_c) / batch
        post_power = cp.mean(post_c * post_c, axis=0, keepdims=True).T
        delta = cp.float32(ltp) * ltp_term - cp.float32(ltd) * ltd_term
        delta -= cp.float32(oja) * post_power * self.W_rec
        delta -= cp.float32(decay) * self.W_rec

        self.W_rec += cp.float32(lr) * delta * self.W_rec_mask
        cp.clip(self.W_rec, -clip, clip, out=self.W_rec)
        self.W_rec *= self.W_rec_mask

        if normalize:
            row_norm = cp.sqrt(cp.sum(self.W_rec * self.W_rec, axis=1, keepdims=True)) + cp.float32(1e-6)
            self.W_rec *= self.W_rec_target_norm / row_norm
            cp.clip(self.W_rec, -clip, clip, out=self.W_rec)
            self.W_rec *= self.W_rec_mask

        if return_stats:
            cp.cuda.Stream.null.synchronize()
            return self.recurrent_weight_stats()
        return None

    def input_weight_stats(self):
        active = self.W_in[self.W_in_mask]
        row_norm = cp.sqrt(cp.sum(self.W_in * self.W_in, axis=1))
        return {
            "active_mean": float(cp.mean(active).get()),
            "active_std": float(cp.std(active).get()),
            "active_min": float(cp.min(active).get()),
            "active_max": float(cp.max(active).get()),
            "row_norm_mean": float(cp.mean(row_norm).get()),
            "row_norm_std": float(cp.std(row_norm).get()),
        }

    def apply_input_plasticity(
        self,
        input_vectors,
        reservoir_rates,
        lr: float | None = None,
        oja: float | None = None,
        decay: float | None = None,
        clip: float | None = None,
        normalize: bool | None = None,
    ):
        """Rate-STDP/Oja update for input -> reservoir synapses.

        This is intentionally unsupervised and local: it uses centered pre activity
        from the input layer and centered post activity from the reservoir rate layer.
        The original locality mask is preserved, so no long-range input synapses appear.
        """
        cfg = self.config
        lr = cfg.input_plastic_lr if lr is None else float(lr)
        oja = cfg.input_plastic_oja if oja is None else float(oja)
        decay = cfg.input_plastic_decay if decay is None else float(decay)
        clip = cfg.input_plastic_clip if clip is None else float(clip)
        normalize = cfg.input_plastic_normalize if normalize is None else bool(normalize)
        if lr == 0:
            return self.input_weight_stats()

        pre = cp.asarray(input_vectors, dtype=cp.float32)
        post = cp.asarray(reservoir_rates, dtype=cp.float32)
        if pre.ndim == 1:
            pre = pre[None, :]
        if post.ndim == 1:
            post = post[None, :]
        if pre.shape[0] != post.shape[0]:
            raise ValueError("input_vectors and reservoir_rates must have the same batch dimension")

        # Center within each example. This avoids learning mostly from the baseline firing rate.
        pre_c = pre - cp.mean(pre, axis=1, keepdims=True)
        post_c = post - cp.mean(post, axis=1, keepdims=True)
        batch = cp.float32(pre.shape[0])

        hebbian = (post_c.T @ pre_c) / batch
        post_power = cp.mean(post_c * post_c, axis=0, keepdims=True).T
        delta = hebbian - cp.float32(oja) * post_power * self.W_in - cp.float32(decay) * self.W_in

        self.W_in += cp.float32(lr) * delta * self.W_in_mask
        cp.clip(self.W_in, -clip, clip, out=self.W_in)
        self.W_in *= self.W_in_mask

        if normalize:
            row_norm = cp.sqrt(cp.sum(self.W_in * self.W_in, axis=1, keepdims=True)) + cp.float32(1e-6)
            self.W_in *= self.W_in_target_norm / row_norm
            cp.clip(self.W_in, -clip, clip, out=self.W_in)
            self.W_in *= self.W_in_mask

        cp.cuda.Stream.null.synchronize()
        return self.input_weight_stats()

    def step_live(
        self,
        input_vector=None,
        envelope_value: float = 1.0,
        amplitude: float | None = None,
        return_transition: bool = False,
    ):
        prev_R = self.R[:, 0].copy() if return_transition else None
        if input_vector is None:
            U = None
        else:
            U = cp.asarray(input_vector.reshape(-1, 1).astype(np.float32)) * cp.float32(envelope_value)
        self.R, self.V, self.I = self.step_state(self.R, self.V, self.I, U=U, amplitude=amplitude)
        self.t += self.config.dt
        self.tick += 1
        if return_transition:
            return prev_R, self.R[:, 0].copy()
        return None

    def reservoir_frame(self):
        return cp.asnumpy(self.R[:, 0].reshape(self.config.reservoir_shape))

    def current_rate(self):
        return cp.asnumpy(self.R[:, 0])


reservoir = MNISTEchoReservoir(RESERVOIR_CONFIG)
print(reservoir.W_in.shape, reservoir.W_rec.shape)

## Readouts

`RidgeProbe` is the batch least-squares baseline. `OnlineRLSProbe` is the main interactive readout; with forgetting set to `1.0`, one pass is an online ridge solution over the examples seen so far. `OnlineSoftmaxProbe` is kept as a lower-level SGD baseline for comparison.


In [ ]:
class RidgeProbe:
    def __init__(self, weights, mean, std):
        self.weights = weights.astype(np.float32)
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)
        self.readout_name = "batch ridge probe"

    @classmethod
    def fit(cls, features, labels, ridge: float = 1e-2):
        features = features.astype(np.float32)
        mean = features.mean(axis=0)
        std = features.std(axis=0) + 1e-6
        X = (features - mean) / std
        X_aug = np.column_stack([X, np.ones(len(X), dtype=np.float32)])
        Y = one_hot(labels)

        lhs = X_aug.T @ X_aug
        lhs.flat[:: lhs.shape[0] + 1] += ridge
        lhs[-1, -1] -= ridge
        rhs = X_aug.T @ Y
        weights = np.linalg.solve(lhs, rhs)
        return cls(weights, mean, std)

    def logits(self, features):
        X = np.asarray(features, dtype=np.float32)
        if X.ndim == 1:
            X = X[None, :]
        X = (X - self.mean) / self.std
        X_aug = np.column_stack([X, np.ones(len(X), dtype=np.float32)])
        return X_aug @ self.weights

    def probabilities(self, features):
        logits = self.logits(features)
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_logits = np.exp(logits)
        return exp_logits / exp_logits.sum(axis=1, keepdims=True)

    def predict(self, features):
        return self.logits(features).argmax(axis=1)


class OnlineRLSProbe:
    """Mini-batch recursive least squares readout for online ESN-style training.

    The update is the Woodbury form of online ridge regression. With forgetting=1.0
    and one pass over the data, this matches a ridge-style linear probe over the
    examples seen so far, but it can also keep adapting during live interaction.
    """

    def __init__(self, n_features: int, delta: float = 1e-2, forgetting: float = 1.0):
        self.n_features = int(n_features)
        self.delta = float(delta)
        self.forgetting = float(forgetting)
        self.mean = np.zeros(self.n_features, dtype=np.float32)
        self.std = np.ones(self.n_features, dtype=np.float32)
        self.weights = np.zeros((self.n_features + 1, 10), dtype=np.float32)
        self.P = np.eye(self.n_features + 1, dtype=np.float32)
        self.updates = 0
        self.examples_seen = 0
        self.last_loss = None
        self.last_batch_acc = None
        self.readout_name = "online RLS probe"
        self.reset_weights()

    def set_normalizer(self, features):
        features = np.asarray(features, dtype=np.float32)
        self.mean = features.mean(axis=0).astype(np.float32)
        self.std = (features.std(axis=0) + 1e-6).astype(np.float32)

    def reset_weights(self, delta: float | None = None):
        if delta is not None:
            self.delta = float(delta)
        prior = max(float(self.delta), 1e-8)
        self.weights = np.zeros((self.n_features + 1, 10), dtype=np.float32)
        self.P = np.eye(self.n_features + 1, dtype=np.float32) / prior
        self.updates = 0
        self.examples_seen = 0
        self.last_loss = None
        self.last_batch_acc = None

    def _standardize(self, features):
        X = np.asarray(features, dtype=np.float32)
        if X.ndim == 1:
            X = X[None, :]
        return (X - self.mean) / self.std

    def _augment(self, features):
        X = self._standardize(features)
        return np.column_stack([X, np.ones(len(X), dtype=np.float32)]).astype(np.float32)

    def logits(self, features):
        return self._augment(features) @ self.weights

    def probabilities(self, features):
        logits = self.logits(features)
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_logits = np.exp(logits)
        return exp_logits / exp_logits.sum(axis=1, keepdims=True)

    def predict(self, features):
        return self.logits(features).argmax(axis=1)

    def partial_fit(self, features, labels, forgetting: float | None = None):
        labels = np.asarray(labels, dtype=np.int64).reshape(-1)
        X_aug = self._augment(features)
        targets = one_hot(labels).astype(np.float32)
        lam = self.forgetting if forgetting is None else float(forgetting)
        lam = float(np.clip(lam, 0.90, 1.0))
        self.forgetting = lam

        # Batch RLS update. Solving in batch is much faster than looping over
        # individual examples, but it is algebraically the same update.
        predictions_before = X_aug @ self.weights
        error = targets - predictions_before
        XP = X_aug @ self.P
        S = XP @ X_aug.T
        S.flat[:: S.shape[0] + 1] += lam
        try:
            K_T = np.linalg.solve(S, XP)
        except np.linalg.LinAlgError:
            jitter = 1e-5 * max(1.0, float(np.trace(S)) / max(1, S.shape[0]))
            S.flat[:: S.shape[0] + 1] += jitter
            K_T = np.linalg.solve(S, XP)

        self.weights += (K_T.T @ error).astype(np.float32)
        self.P = ((self.P - K_T.T @ XP) / lam).astype(np.float32)
        self.P = (0.5 * (self.P + self.P.T)).astype(np.float32)

        predictions_after = X_aug @ self.weights
        loss = float(np.mean((targets - predictions_after) ** 2))
        acc = float(np.mean(predictions_after.argmax(axis=1) == labels))
        self.last_loss = loss
        self.last_batch_acc = acc
        self.updates += 1
        self.examples_seen += len(labels)
        return loss, acc

    def train_epochs(
        self,
        features,
        labels,
        epochs: int,
        batch_size: int,
        seed: int = 0,
        progress_desc: str | None = None,
    ):
        rng = np.random.default_rng(seed)
        labels = np.asarray(labels, dtype=np.int64)
        history = []
        epoch_iter = range(int(epochs))
        if progress_desc is not None:
            epoch_iter = tqdm(epoch_iter, total=int(epochs), desc=progress_desc, leave=False)
        for epoch in epoch_iter:
            order = rng.permutation(len(labels))
            batch_iter = range(0, len(order), int(batch_size))
            if progress_desc is not None:
                batch_iter = tqdm(
                    batch_iter,
                    total=(len(order) + int(batch_size) - 1) // int(batch_size),
                    desc=f"{progress_desc} epoch {epoch + 1}",
                    leave=False,
                )
            for start in batch_iter:
                batch_idx = order[start : start + int(batch_size)]
                self.partial_fit(features[batch_idx], labels[batch_idx])
            history.append((epoch + 1, self.last_loss, self.last_batch_acc))
        return history


class OnlineSoftmaxProbe:
    def __init__(self, n_features: int, lr: float = 5e-2, l2: float = 1e-4):
        self.n_features = int(n_features)
        self.lr = float(lr)
        self.l2 = float(l2)
        self.mean = np.zeros(self.n_features, dtype=np.float32)
        self.std = np.ones(self.n_features, dtype=np.float32)
        self.weights = np.zeros((self.n_features + 1, 10), dtype=np.float32)
        self.updates = 0
        self.examples_seen = 0
        self.last_loss = None
        self.last_batch_acc = None
        self.readout_name = "online softmax probe"

    def set_normalizer(self, features):
        features = np.asarray(features, dtype=np.float32)
        self.mean = features.mean(axis=0).astype(np.float32)
        self.std = (features.std(axis=0) + 1e-6).astype(np.float32)

    def reset_weights(self):
        self.weights.fill(0.0)
        self.updates = 0
        self.examples_seen = 0
        self.last_loss = None
        self.last_batch_acc = None

    def _standardize(self, features):
        X = np.asarray(features, dtype=np.float32)
        if X.ndim == 1:
            X = X[None, :]
        return (X - self.mean) / self.std

    def _augment(self, features):
        X = self._standardize(features)
        return np.column_stack([X, np.ones(len(X), dtype=np.float32)])

    def logits(self, features):
        return self._augment(features) @ self.weights

    def probabilities(self, features):
        logits = self.logits(features)
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_logits = np.exp(logits)
        return exp_logits / exp_logits.sum(axis=1, keepdims=True)

    def predict(self, features):
        return self.logits(features).argmax(axis=1)

    def partial_fit(self, features, labels, lr: float | None = None, l2: float | None = None):
        labels = np.asarray(labels, dtype=np.int64).reshape(-1)
        X_aug = self._augment(features)
        step_lr = self.lr if lr is None else float(lr)
        step_l2 = self.l2 if l2 is None else float(l2)

        logits = X_aug @ self.weights
        logits = logits - logits.max(axis=1, keepdims=True)
        probs = np.exp(logits)
        probs /= probs.sum(axis=1, keepdims=True)
        targets = one_hot(labels)

        grad = X_aug.T @ (probs - targets) / len(labels)
        grad[:-1] += step_l2 * self.weights[:-1]
        self.weights -= step_lr * grad.astype(np.float32)

        loss = float(-(targets * np.log(probs + 1e-8)).sum(axis=1).mean())
        acc = float(np.mean(probs.argmax(axis=1) == labels))
        self.last_loss = loss
        self.last_batch_acc = acc
        self.updates += 1
        self.examples_seen += len(labels)
        return loss, acc

    def train_epochs(
        self,
        features,
        labels,
        epochs: int,
        batch_size: int,
        seed: int = 0,
        progress_desc: str | None = None,
    ):
        rng = np.random.default_rng(seed)
        labels = np.asarray(labels, dtype=np.int64)
        history = []
        epoch_iter = range(int(epochs))
        if progress_desc is not None:
            epoch_iter = tqdm(epoch_iter, total=int(epochs), desc=progress_desc, leave=False)
        for epoch in epoch_iter:
            order = rng.permutation(len(labels))
            batch_iter = range(0, len(order), int(batch_size))
            if progress_desc is not None:
                batch_iter = tqdm(
                    batch_iter,
                    total=(len(order) + int(batch_size) - 1) // int(batch_size),
                    desc=f"{progress_desc} epoch {epoch + 1}",
                    leave=False,
                )
            for start in batch_iter:
                batch_idx = order[start : start + int(batch_size)]
                self.partial_fit(features[batch_idx], labels[batch_idx])
            history.append((epoch + 1, self.last_loss, self.last_batch_acc))
        return history


def softmax_accuracy(probe, features, labels):
    return float(np.mean(probe.predict(features) == np.asarray(labels)))


## Feature Extraction And Probe Smoke Tests

The ridge readout is the closed-form baseline. The online RLS probe below starts from the same ridge prior and is trained with the same mini-batch update path used by the live controls. The softmax probe is kept to show how much of the earlier online-learning gap came from SGD optimization rather than the reservoir itself.


In [ ]:
train_images, train_labels = select_balanced(
    train_images_all,
    train_labels_all,
    PROBE_CONFIG.train_per_digit,
    seed=11,
)
test_images, test_labels = select_balanced(
    test_images_all,
    test_labels_all,
    PROBE_CONFIG.test_per_digit,
    seed=12,
)

X_train_input = input_from_images(train_images)
X_test_input = input_from_images(test_images)

start = time.perf_counter()
train_features = reservoir.collect_features(
    X_train_input,
    batch_size=PROBE_CONFIG.batch_size,
    amplitude=RESERVOIR_CONFIG.amplitude,
    progress_desc="train features",
)
test_features = reservoir.collect_features(
    X_test_input,
    batch_size=PROBE_CONFIG.batch_size,
    amplitude=RESERVOIR_CONFIG.amplitude,
    progress_desc="test features",
)
feature_seconds = time.perf_counter() - start

ridge_probe = RidgeProbe.fit(train_features, train_labels, ridge=PROBE_CONFIG.ridge)
ridge_train_acc = softmax_accuracy(ridge_probe, train_features, train_labels)
ridge_test_acc = softmax_accuracy(ridge_probe, test_features, test_labels)

rls_probe = OnlineRLSProbe(
    reservoir.n_reservoir,
    delta=PROBE_CONFIG.rls_delta,
    forgetting=PROBE_CONFIG.rls_forgetting,
)
rls_probe.set_normalizer(train_features)
rls_probe.train_epochs(
    train_features,
    train_labels,
    epochs=PROBE_CONFIG.rls_epochs,
    batch_size=PROBE_CONFIG.rls_batch_size,
    seed=41,
    progress_desc="online RLS",
)
rls_train_acc = softmax_accuracy(rls_probe, train_features, train_labels)
rls_test_acc = softmax_accuracy(rls_probe, test_features, test_labels)

online_probe = OnlineSoftmaxProbe(
    reservoir.n_reservoir,
    lr=PROBE_CONFIG.online_lr,
    l2=PROBE_CONFIG.online_l2,
)
online_probe.set_normalizer(train_features)
online_probe.train_epochs(
    train_features,
    train_labels,
    epochs=PROBE_CONFIG.online_epochs,
    batch_size=PROBE_CONFIG.online_batch_size,
    seed=31,
    progress_desc="online softmax",
)
online_train_acc = softmax_accuracy(online_probe, train_features, train_labels)
online_test_acc = softmax_accuracy(online_probe, test_features, test_labels)

print(f"feature extraction: {feature_seconds:.3f}s")
print(f"ridge baseline train/test: {ridge_train_acc:.3f} / {ridge_test_acc:.3f}")
print(f"online RLS train/test: {rls_train_acc:.3f} / {rls_test_acc:.3f}")
print(f"online softmax train/test: {online_train_acc:.3f} / {online_test_acc:.3f}")
print(f"RLS updates: {rls_probe.updates}; examples seen: {rls_probe.examples_seen}")
print(f"softmax updates: {online_probe.updates}; examples seen: {online_probe.examples_seen}")
print(
    "feature stats:",
    f"mean={train_features.mean():.6f}",
    f"std={train_features.std():.6f}",
    f"min={train_features.min():.6f}",
    f"max={train_features.max():.6f}",
)
print("input synapse stats:", reservoir.input_weight_stats())
print("recurrent synapse stats:", reservoir.recurrent_weight_stats())
print("cell dynamic stats:", reservoir.cell_dynamic_stats())


In [ ]:
from plotly.subplots import make_subplots


def confusion_matrix_counts(labels, predictions, classes: int = 10):
    labels = np.asarray(labels, dtype=np.int64)
    predictions = np.asarray(predictions, dtype=np.int64)
    matrix = np.zeros((classes, classes), dtype=np.int64)
    np.add.at(matrix, (labels, predictions), 1)
    return matrix


def plot_confusion_matrices(probe=rls_probe, name: str | None = None):
    if name is None:
        name = getattr(probe, "readout_name", probe.__class__.__name__)
    train_pred = probe.predict(train_features)
    test_pred = probe.predict(test_features)
    train_cm = confusion_matrix_counts(train_labels, train_pred)
    test_cm = confusion_matrix_counts(test_labels, test_pred)

    train_acc = np.trace(train_cm) / np.maximum(1, train_cm.sum())
    test_acc = np.trace(test_cm) / np.maximum(1, test_cm.sum())
    digits = list(range(10))

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            f"Train confusion ({train_acc:.3f})",
            f"Test confusion ({test_acc:.3f})",
        ),
        horizontal_spacing=0.12,
    )

    heatmap_kwargs = dict(
        x=digits,
        y=digits,
        colorscale="Blues",
        texttemplate="%{z}",
        hovertemplate="true=%{y}<br>pred=%{x}<br>count=%{z}<extra></extra>",
        colorbar=dict(title="count"),
    )
    fig.add_trace(go.Heatmap(z=train_cm, **heatmap_kwargs), row=1, col=1)
    fig.add_trace(go.Heatmap(z=test_cm, **heatmap_kwargs, showscale=False), row=1, col=2)

    fig.update_xaxes(title_text="Predicted digit", dtick=1)
    fig.update_yaxes(title_text="True digit", dtick=1, autorange="reversed")
    fig.update_layout(
        title=f"Full confusion matrices for {name}",
        width=1000,
        height=460,
        margin=dict(l=40, r=20, t=70, b=45),
    )
    fig.show()
    return train_cm, test_cm


# Change this to ridge_probe or online_probe if you want to inspect another readout.
train_confusion, test_confusion = plot_confusion_matrices(rls_probe)


## Live Online Learning Controls

Digit buttons present one randomly selected image. If `train on presented image` is checked, the online readout updates from that example after the readout window. The batch controls run online RLS on fresh random MNIST training examples; recurrent and input plasticity can be enabled separately.


In [ ]:
class LiveMNISTEchoView:
    def __init__(self, reservoir: MNISTEchoReservoir, probe):
        self.reservoir = reservoir
        self.probe = probe
        self.rng = np.random.default_rng(123)
        self._stop = threading.Event()
        self._train_stop = threading.Event()
        self._thread = None
        self._train_thread = None
        self._lock = threading.RLock()

        self.current_digit = None
        self.current_image = np.zeros((28, 28), dtype=np.uint8)
        self.current_input = np.zeros(self.reservoir.n_input, dtype=np.float32)
        self.readout_sum = np.zeros(self.reservoir.n_reservoir, dtype=np.float32)
        self.readout_count = 0
        self.sequence_step = 0
        self.last_eval_acc = None

        self.input_fig = go.FigureWidget(
            data=[go.Heatmap(z=np.zeros(self.reservoir.config.input_shape), colorscale="Gray", zmin=0, zmax=1, showscale=False)]
        )
        self.input_fig.update_layout(
            width=310,
            height=310,
            margin=dict(l=0, r=0, b=0, t=24),
            title=dict(text="14x14 input layer", x=0.5, y=0.98, font=dict(size=13)),
            xaxis=dict(visible=False, fixedrange=True),
            yaxis=dict(visible=False, fixedrange=True, autorange="reversed"),
        )

        self.reservoir_fig = go.FigureWidget(
            data=[go.Heatmap(z=np.zeros(self.reservoir.config.reservoir_shape), colorscale="Viridis", zmin=0, zmax=0.2, showscale=False)]
        )
        self.reservoir_fig.update_layout(
            width=520,
            height=520,
            margin=dict(l=0, r=0, b=0, t=24),
            title=dict(text="28x28 reservoir rates", x=0.5, y=0.98, font=dict(size=13)),
            xaxis=dict(visible=False, fixedrange=True),
            yaxis=dict(visible=False, fixedrange=True, autorange="reversed"),
        )

        self.probe_fig = go.FigureWidget(
            data=[go.Bar(x=list(range(10)), y=np.full(10, 0.1, dtype=np.float32), marker_color="#4c78a8")]
        )
        self.probe_fig.update_layout(
            width=640,
            height=260,
            margin=dict(l=30, r=10, b=30, t=24),
            title=dict(text="readout probabilities", x=0.5, y=0.98, font=dict(size=13)),
            yaxis=dict(range=[0, 1], fixedrange=True),
            xaxis=dict(dtick=1, fixedrange=True),
        )

        self.digit_buttons = [widgets.Button(description=str(d), layout=widgets.Layout(width="42px")) for d in range(10)]
        for digit, button in enumerate(self.digit_buttons):
            button.on_click(lambda _, d=digit: self.present_digit(d))

        self.train_batch_button = widgets.Button(description="Train Batch", icon="graduation-cap", button_style="info")
        self.auto_train_button = widgets.Button(description="Auto Train", icon="play", button_style="success")
        self.stop_button = widgets.Button(description="Stop", icon="stop", button_style="danger")
        self.reset_state_button = widgets.Button(description="Reset State", icon="refresh")
        self.resample_dynamics_button = widgets.Button(description="Resample Dynamics", icon="sliders")
        self.reset_probe_button = widgets.Button(description="Reset Probe", icon="eraser")
        self.eval_button = widgets.Button(description="Eval", icon="check")
        self.plasticity_button = widgets.Button(description="Recurrent Plastic Batch", icon="random")

        self.train_batch_button.on_click(lambda _: self.train_one_batch_async())
        self.auto_train_button.on_click(lambda _: self.start_auto_train())
        self.stop_button.on_click(lambda _: self.stop_all())
        self.reset_state_button.on_click(lambda _: self.reset_state())
        self.resample_dynamics_button.on_click(lambda _: self.resample_dynamics())
        self.reset_probe_button.on_click(lambda _: self.reset_probe())
        self.eval_button.on_click(lambda _: self.evaluate_async())
        self.plasticity_button.on_click(lambda _: self.plasticity_one_batch_async())

        self.sample_source = widgets.Dropdown(options=["train", "test"], value="train", description="digit source")
        self.train_on_present = widgets.Checkbox(value=True, description="train readout on presented image")
        self.rec_plastic_on_present = widgets.Checkbox(value=True, description="recurrent STDP on presented image")
        self.rec_plastic_on_batch = widgets.Checkbox(value=True, description="recurrent STDP during batches")
        self.input_plastic_on_present = widgets.Checkbox(value=False, description="input STDP on presented image")
        self.input_plastic_on_batch = widgets.Checkbox(value=False, description="input STDP during batches")
        self.reset_before_image = widgets.Checkbox(value=True, description="reset before image")
        self.amplitude = widgets.FloatSlider(value=self.reservoir.config.amplitude, min=0.0, max=10.0, step=0.1, description="amplitude", continuous_update=False, style={"description_width": "initial"})
        self.pre_steps = widgets.IntSlider(value=self.reservoir.config.pre_steps, min=0, max=50, description="pre", continuous_update=False)
        self.on_steps = widgets.IntSlider(value=self.reservoir.config.on_steps, min=1, max=120, description="on", continuous_update=False)
        self.off_steps = widgets.IntSlider(value=self.reservoir.config.off_steps, min=0, max=80, description="off", continuous_update=False)
        self.readout_start = widgets.IntSlider(value=self.reservoir.config.readout_start, min=0, max=180, description="readout start", continuous_update=False, style={"description_width": "initial"})
        self.steps_per_frame = widgets.IntSlider(value=1, min=1, max=20, description="steps/frame", continuous_update=False, style={"description_width": "initial"})
        self.delay_ms = widgets.IntSlider(value=20, min=0, max=200, description="delay ms", continuous_update=False)
        self.reservoir_zmax = widgets.FloatSlider(value=0.2, min=0.02, max=2.0, step=0.02, description="rate zmax", continuous_update=False, style={"description_width": "initial"})
        self.online_lr = widgets.FloatLogSlider(value=float(getattr(self.probe, "lr", PROBE_CONFIG.online_lr)), base=10, min=-10, max=0, step=0.1, description="softmax lr", continuous_update=False, style={"description_width": "initial"})
        self.online_l2 = widgets.FloatLogSlider(value=float(getattr(self.probe, "l2", PROBE_CONFIG.online_l2)), base=10, min=-7, max=2, step=0.25, description="softmax L2", continuous_update=False, style={"description_width": "initial"})
        self.rls_delta = widgets.FloatLogSlider(value=float(getattr(self.probe, "delta", PROBE_CONFIG.rls_delta)), base=10, min=-6, max=2, step=0.25, description="RLS delta", continuous_update=False, style={"description_width": "initial"})
        self.rls_forgetting = widgets.FloatSlider(value=float(getattr(self.probe, "forgetting", PROBE_CONFIG.rls_forgetting)), min=0.90, max=1.0, step=0.001, readout_format=".3f", description="RLS forget", continuous_update=False, style={"description_width": "initial"})
        if not hasattr(self.probe, "lr"):
            self.online_lr.layout.display = "none"
        if not hasattr(self.probe, "l2"):
            self.online_l2.layout.display = "none"
        if not hasattr(self.probe, "delta"):
            self.rls_delta.layout.display = "none"
        if not hasattr(self.probe, "forgetting"):
            self.rls_forgetting.layout.display = "none"
        self.rec_plastic_lr = widgets.FloatLogSlider(value=self.reservoir.config.rec_plastic_lr, base=10, min=-7, max=-2, step=0.25, description="rec STDP lr", continuous_update=False, style={"description_width": "initial"})
        self.rec_plastic_ltp = widgets.FloatSlider(value=self.reservoir.config.rec_plastic_ltp, min=0.0, max=2.0, step=0.05, description="rec A+", continuous_update=False, style={"description_width": "initial"})
        self.rec_plastic_ltd = widgets.FloatSlider(value=self.reservoir.config.rec_plastic_ltd, min=0.0, max=2.0, step=0.05, description="rec A-", continuous_update=False, style={"description_width": "initial"})
        self.rec_plastic_oja = widgets.FloatLogSlider(value=self.reservoir.config.rec_plastic_oja, base=10, min=-2, max=2, step=0.25, description="rec Oja", continuous_update=False, style={"description_width": "initial"})
        self.rec_plastic_decay = widgets.FloatLogSlider(value=self.reservoir.config.rec_plastic_decay, base=10, min=-7, max=-2, step=0.25, description="rec decay", continuous_update=False, style={"description_width": "initial"})
        self.input_plastic_lr = widgets.FloatLogSlider(value=self.reservoir.config.input_plastic_lr, base=10, min=-6, max=-1, step=0.25, description="input STDP lr", continuous_update=False, style={"description_width": "initial"})
        self.input_plastic_oja = widgets.FloatLogSlider(value=self.reservoir.config.input_plastic_oja, base=10, min=-2, max=2, step=0.25, description="input Oja", continuous_update=False, style={"description_width": "initial"})
        self.input_plastic_decay = widgets.FloatLogSlider(value=self.reservoir.config.input_plastic_decay, base=10, min=-7, max=-2, step=0.25, description="input decay", continuous_update=False, style={"description_width": "initial"})
        self.train_batch_size = widgets.IntSlider(value=64, min=1, max=2048, step=1, description="batch", continuous_update=False)
        self.auto_delay_ms = widgets.IntSlider(value=50, min=0, max=1000, step=10, description="train delay", continuous_update=False, style={"description_width": "initial"})
        self.eval_per_digit = widgets.IntSlider(value=20, min=5, max=200, step=5, description="eval/digit", continuous_update=False, style={"description_width": "initial"})
        self.status = widgets.HTML(value="idle")
        self.reservoir_zmax.observe(self._update_zmax, names="value")

    def _update_zmax(self, change=None):
        with self.reservoir_fig.batch_update():
            self.reservoir_fig.data[0].zmax = float(self.reservoir_zmax.value)

    def _sample_digit(self, digit: int):
        if self.sample_source.value == "train":
            images, labels = train_images_all, train_labels_all
        else:
            images, labels = test_images_all, test_labels_all
        choices = np.flatnonzero(labels == digit)
        idx = int(self.rng.choice(choices))
        image = images[idx]
        input_vec = input_from_images(image[None, :, :])[0]
        return image, input_vec

    def _total_steps(self):
        return int(self.pre_steps.value + self.on_steps.value + self.off_steps.value)

    def _current_envelope_and_input(self):
        pre = int(self.pre_steps.value)
        on = int(self.on_steps.value)
        if pre <= self.sequence_step < pre + on:
            pulse_step = self.sequence_step - pre
            env = self.reservoir.envelope(pulse_step, on)
            return env, self.current_input
        return 0.0, None

    def _current_feature(self):
        if self.readout_count > 0:
            return self.readout_sum / self.readout_count
        return self.reservoir.current_rate()

    def _draw(self, env):
        feature = self._current_feature()
        probs = self.probe.probabilities(feature)[0]
        pred = int(np.argmax(probs))
        input_grid = (self.current_input * env).reshape(self.reservoir.config.input_shape)
        reservoir_grid = self.reservoir.reservoir_frame()

        with self.input_fig.batch_update():
            self.input_fig.data[0].z = input_grid
        with self.reservoir_fig.batch_update():
            self.reservoir_fig.data[0].z = reservoir_grid
            self.reservoir_fig.data[0].zmax = float(self.reservoir_zmax.value)
        with self.probe_fig.batch_update():
            self.probe_fig.data[0].y = probs
            colors = ["#4c78a8"] * 10
            colors[pred] = "#f58518"
            self.probe_fig.data[0].marker.color = colors

        true_text = "?" if self.current_digit is None else str(self.current_digit)
        loss = "n/a" if self.probe.last_loss is None else f"{self.probe.last_loss:.3f}"
        bacc = "n/a" if self.probe.last_batch_acc is None else f"{self.probe.last_batch_acc:.3f}"
        eval_text = "n/a" if self.last_eval_acc is None else f"{self.last_eval_acc:.3f}"
        self.status.value = (
            f"true={true_text} pred={pred} p={probs[pred]:.3f} "
            f"step={self.sequence_step}/{self._total_steps()} readout_n={self.readout_count} | "
            f"updates={self.probe.updates} seen={self.probe.examples_seen} "
            f"loss={loss} batch_acc={bacc} eval={eval_text}"
        )

    def _sync_probe_hyperparams(self):
        if hasattr(self.probe, "lr"):
            self.probe.lr = float(self.online_lr.value)
        if hasattr(self.probe, "l2"):
            self.probe.l2 = float(self.online_l2.value)
        if hasattr(self.probe, "delta"):
            self.probe.delta = float(self.rls_delta.value)
        if hasattr(self.probe, "forgetting"):
            self.probe.forgetting = float(self.rls_forgetting.value)

    def _apply_recurrent_plasticity(self, pre_rates, post_rates, return_stats: bool = False):
        return self.reservoir.apply_recurrent_plasticity(
            pre_rates,
            post_rates,
            lr=float(self.rec_plastic_lr.value),
            ltp=float(self.rec_plastic_ltp.value),
            ltd=float(self.rec_plastic_ltd.value),
            oja=float(self.rec_plastic_oja.value),
            decay=float(self.rec_plastic_decay.value),
            return_stats=return_stats,
        )

    def _apply_input_plasticity(self, input_vectors, features):
        return self.reservoir.apply_input_plasticity(
            input_vectors,
            features,
            lr=float(self.input_plastic_lr.value),
            oja=float(self.input_plastic_oja.value),
            decay=float(self.input_plastic_decay.value),
        )

    def _run_sequence(self):
        self.readout_sum.fill(0.0)
        self.readout_count = 0
        self.sequence_step = 0
        total = self._total_steps()
        readout_start = int(self.readout_start.value)

        while not self._stop.is_set() and self.sequence_step < total:
            with self._lock:
                for _ in range(int(self.steps_per_frame.value)):
                    if self.sequence_step >= total:
                        break
                    env, input_vec = self._current_envelope_and_input()
                    transition = self.reservoir.step_live(
                        input_vec,
                        envelope_value=env,
                        amplitude=float(self.amplitude.value),
                        return_transition=bool(self.rec_plastic_on_present.value),
                    )
                    if self.rec_plastic_on_present.value and transition is not None:
                        pre_rate, post_rate = transition
                        self._apply_recurrent_plasticity(pre_rate, post_rate)
                    if self.sequence_step >= readout_start:
                        self.readout_sum += self.reservoir.current_rate()
                        self.readout_count += 1
                    self.sequence_step += 1
                self._draw(env)
            delay = int(self.delay_ms.value) / 1000.0
            if delay:
                time.sleep(delay)

        with self._lock:
            if self.current_digit is not None and self.readout_count > 0:
                feature = self._current_feature()
                if self.train_on_present.value:
                    self._sync_probe_hyperparams()
                    self.probe.partial_fit(feature, [self.current_digit])
                if self.input_plastic_on_present.value:
                    self._apply_input_plasticity(self.current_input, feature)
            self._draw(0.0)

    def present_digit(self, digit: int):
        self.stop_presentation()
        with self._lock:
            self.current_digit = int(digit)
            self.current_image, self.current_input = self._sample_digit(digit)
            if self.reset_before_image.value:
                self.reservoir.reset_state(batch_size=1)
        self._stop.clear()
        self._thread = threading.Thread(target=self._run_sequence, daemon=True, name="mnist-echo-present")
        self._thread.start()

    def _training_features_from_random_batch(self):
        images, labels = sample_random_labeled(
            train_images_all,
            train_labels_all,
            int(self.train_batch_size.value),
            self.rng,
        )
        input_vectors = input_from_images(images)
        features = self.reservoir.collect_features(
            input_vectors,
            batch_size=min(int(self.train_batch_size.value), 256),
            amplitude=float(self.amplitude.value),
            pre_steps=int(self.pre_steps.value),
            on_steps=int(self.on_steps.value),
            off_steps=int(self.off_steps.value),
            readout_start=int(self.readout_start.value),
        )
        return features, labels, input_vectors

    def train_one_batch(self):
        with self._lock:
            self._sync_probe_hyperparams()
            features, labels, input_vectors = self._training_features_from_random_batch()
            self.probe.partial_fit(features, labels)
            if self.rec_plastic_on_batch.value:
                self._apply_recurrent_plasticity(features, features)
            if self.input_plastic_on_batch.value:
                self._apply_input_plasticity(input_vectors, features)
            self._draw(0.0)

    def train_one_batch_async(self):
        threading.Thread(target=self.train_one_batch, daemon=True, name="mnist-echo-train-batch").start()

    def plasticity_one_batch(self):
        with self._lock:
            features, labels, input_vectors = self._training_features_from_random_batch()
            self._apply_recurrent_plasticity(features, features)
            if self.input_plastic_on_batch.value:
                self._apply_input_plasticity(input_vectors, features)
            self._draw(0.0)

    def plasticity_one_batch_async(self):
        threading.Thread(target=self.plasticity_one_batch, daemon=True, name="mnist-echo-plastic-batch").start()

    def _auto_train_loop(self):
        while not self._train_stop.is_set():
            self.train_one_batch()
            delay = int(self.auto_delay_ms.value) / 1000.0
            if delay:
                time.sleep(delay)

    def start_auto_train(self):
        if self._train_thread is not None and self._train_thread.is_alive():
            return
        self._train_stop.clear()
        self._train_thread = threading.Thread(target=self._auto_train_loop, daemon=True, name="mnist-echo-auto-train")
        self._train_thread.start()

    def evaluate(self):
        with self._lock:
            images, labels = sample_balanced(
                test_images_all,
                test_labels_all,
                int(self.eval_per_digit.value),
                self.rng,
            )
            features = self.reservoir.collect_features(
                input_from_images(images),
                batch_size=128,
                amplitude=float(self.amplitude.value),
                pre_steps=int(self.pre_steps.value),
                on_steps=int(self.on_steps.value),
                off_steps=int(self.off_steps.value),
                readout_start=int(self.readout_start.value),
            )
            self.last_eval_acc = softmax_accuracy(self.probe, features, labels)
            self._draw(0.0)

    def evaluate_async(self):
        threading.Thread(target=self.evaluate, daemon=True, name="mnist-echo-eval").start()

    def stop_presentation(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2)

    def stop_training(self):
        self._train_stop.set()
        if self._train_thread is not None:
            self._train_thread.join(timeout=2)

    def stop_all(self):
        self.stop_presentation()
        self.stop_training()

    def reset_state(self):
        self.stop_presentation()
        with self._lock:
            self.reservoir.reset_state(batch_size=1)
            self.current_digit = None
            self.current_input.fill(0.0)
            self.readout_sum.fill(0.0)
            self.readout_count = 0
            self.sequence_step = 0
            self._draw(0.0)

    def reset_probe(self):
        with self._lock:
            self._sync_probe_hyperparams()
            self.probe.reset_weights()
            self.last_eval_acc = None
            self._draw(0.0)

    def resample_dynamics(self):
        self.stop_presentation()
        with self._lock:
            stats = self.reservoir.resample_cell_dynamics(seed=int(self.rng.integers(0, 2**31 - 1)))
            self.reservoir.reset_state(batch_size=1)
            self._draw(0.0)
            self.status.value = f"resampled cell dynamics: {stats}"

    def display(self):
        controls = widgets.VBox([
            widgets.HBox(self.digit_buttons + [self.train_batch_button, self.plasticity_button, self.auto_train_button, self.stop_button]),
            widgets.HBox([self.reset_state_button, self.resample_dynamics_button, self.reset_probe_button, self.eval_button, self.sample_source, self.train_on_present, self.rec_plastic_on_present, self.rec_plastic_on_batch, self.reset_before_image]),
            widgets.HBox([self.input_plastic_on_present, self.input_plastic_on_batch]),
            widgets.HBox([self.amplitude, self.pre_steps, self.on_steps, self.off_steps, self.readout_start]),
            widgets.HBox([self.steps_per_frame, self.delay_ms, self.reservoir_zmax]),
            widgets.HBox([self.train_batch_size, self.rls_delta, self.rls_forgetting, self.online_lr, self.online_l2]),
            widgets.HBox([self.rec_plastic_lr, self.rec_plastic_ltp, self.rec_plastic_ltd]),
            widgets.HBox([self.rec_plastic_oja, self.rec_plastic_decay, self.input_plastic_lr, self.input_plastic_oja, self.input_plastic_decay]),
            widgets.HBox([self.auto_delay_ms, self.eval_per_digit]),
            self.status,
        ])
        display(widgets.VBox([
            controls,
            widgets.HBox([self.input_fig, self.reservoir_fig]),
            self.probe_fig,
        ]))


try:
    live_mnist.stop_all()
except NameError:
    pass

reservoir.reset_state(batch_size=1)
live_mnist = LiveMNISTEchoView(reservoir, rls_probe)
live_mnist.display()


## Manual Experiments

Useful one-liners while tuning. These are intentionally commented out.

In [ ]:
# Stop all live background threads:
# live_mnist.stop_all()

# Reset the online RLS readout, keeping the feature normalizer:
# rls_probe.reset_weights()

# Train the online RLS readout for another pass over already-extracted features.
# With forgetting=1.0, repeated passes count the same examples again, so use this deliberately.
# rls_probe.train_epochs(train_features, train_labels, epochs=1, batch_size=64, seed=99)

# Compare the softmax-SGD baseline after more passes:
# online_probe.reset_weights()
# online_probe.train_epochs(train_features, train_labels, epochs=3, batch_size=64, seed=99)

# Rebuild reservoir after changing structural parameters:
# RESERVOIR_CONFIG.spectral_radius = 0.95
# reservoir = MNISTEchoReservoir(RESERVOIR_CONFIG)

# Present a digit programmatically:
# live_mnist.present_digit(7)

# Tune per-cell heterogeneity before rebuilding the reservoir:
# RESERVOIR_CONFIG.Eta_std = 0.5
# RESERVOIR_CONFIG.current_decay_std = 0.05
# reservoir = MNISTEchoReservoir(RESERVOIR_CONFIG)
